# 6. LangChain vs LangGraph vs CrewAI — Comparison

## Overview

The three frameworks are not competitors so much as three different levels of control. In this
notebook we implement **the same task** in each one and compare the code, the cost, and the
failure modes.

**The task:** given a Medicare benefits topic, produce a short plain-language explanation for a
member — then check it against a member communications standard, and revise it if the check
fails.

That task is deliberately chosen for two reasons. It contains a **loop**, and loops are exactly
where the three frameworks diverge. And the loop is a **compliance check**, which is the shape
most regulated workflows take: produce something, verify it against a standard, do not release
it until it passes.


#### 0. Setup

In [1]:
# ---------------------------------------------------------------------------
# SageMaker Studio setup
#
# There is no .env file and no access key in this lab. boto3 automatically
# picks up short-lived credentials from the SageMaker execution role attached
# to this space, which is both simpler and safer than a long-lived key pair.
# ---------------------------------------------------------------------------
import os
import boto3

# Leave this as None to use the region this SageMaker space is running in.
# Set it ONLY if you enabled Bedrock model access in a different region.
BEDROCK_REGION = None

AWS_REGION = BEDROCK_REGION or boto3.session.Session().region_name or "us-east-1"
CHAT_MODEL_ID = os.environ.get("BEDROCK_CHAT_MODEL_ID", "amazon.nova-pro-v1:0")

import time

from langchain_aws import ChatBedrockConverse

# CrewAI / LiteLLM look for these exact names
os.environ["AWS_REGION_NAME"] = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

session = boto3.Session(region_name=AWS_REGION)

# Show which IAM identity this notebook is using. On SageMaker this is the
# execution role of your space - the role that needs Bedrock permissions.
identity = session.client("sts").get_caller_identity()
EXECUTION_ROLE_ARN = identity["Arn"]

model = ChatBedrockConverse(
    model=CHAT_MODEL_ID,
    client=session.client("bedrock-runtime"),
    temperature=0,
    max_tokens=1024,
)

TOPIC = "what to do when a claim is denied for missing prior authorisation"

# The standard every draft is checked against. Writing this down once, and
# reusing it across all three implementations, is what makes the comparison fair.
STANDARD = (
    "FAIL the draft if any of these are true: it is vague about what the member must do; "
    "it omits the appeal deadline; it contains anything that could be read as clinical advice; "
    "it uses insurance jargon without explaining it; or it does not say how to reach a person."
)

timings = {}

print("Region         :", AWS_REGION)
print("Execution role :", EXECUTION_ROLE_ARN)
print("Ready. Topic under test:", TOPIC)

Region         : us-east-1
Execution role : arn:aws:sts::442042521384:assumed-role/AmazonSageMaker-ExecutionRole-20260807T102279/SageMaker
Ready. Topic under test: what to do when a claim is denied for missing prior authorisation


---
## 1. LangChain implementation

LangChain expresses this as a chain. The catch: **a chain cannot loop**. To add the
revise-if-failed behaviour you write the loop in ordinary Python around the chain.

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

write = ChatPromptTemplate.from_template(
    "Write a 120-word plain-language explanation for a health plan member about {topic}. "
    "State what they must do, and state any deadline explicitly. No clinical advice. "
    "End with how to reach a person."
) | model | StrOutputParser()

check = ChatPromptTemplate.from_template(
    "You review member communications against this standard:\n{standard}\n\n"
    "Reply with exactly one word, PASS or FAIL.\n\nDraft:\n{draft}"
) | model | StrOutputParser()

revise = ChatPromptTemplate.from_template(
    "This member communication failed review against the standard below. Rewrite it so it "
    "passes, keeping it to 120 words.\n\nStandard:\n{standard}\n\nDraft:\n{draft}"
) | model | StrOutputParser()

start = time.time()

draft = write.invoke({"topic": TOPIC})
for attempt in range(2):                      # the loop lives in Python, not in LangChain
    verdict = check.invoke({"draft": draft, "standard": STANDARD}).strip().upper()
    print(f"attempt {attempt + 1}: {verdict}")
    if verdict.startswith("PASS"):
        break
    draft = revise.invoke({"draft": draft, "standard": STANDARD})

timings["LangChain"] = time.time() - start
print("\n--- LANGCHAIN RESULT ---\n", draft)

attempt 1: PASS

--- LANGCHAIN RESULT ---
 If your claim was denied because you didn't get prior authorisation, here's what to do: First, read the denial letter carefully. Note the reason for the denial and any deadlines mentioned. You must request a review of your claim within 60 days of receiving the denial letter. To do this, call our customer service at 1-800-555-HELP and ask for a "reconsideration request form." Fill out the form and submit it along with any additional information that supports your claim. If you need help finding the form or completing it, our customer service team is available Monday to Friday, 8 AM to 6 PM, to assist you.


**What that felt like:** fast to write, easy to read, and the control flow is plain
Python. But the loop counter, the retry limit and the state all live outside the framework —
so nothing is checkpointed, nothing is inspectable, and there is no way to resume a failed run
halfway through.

For a member communication that has to evidence its own review, that last point is the problem.
If someone asks in six months why this letter was approved, you have the final text and nothing
else. The failed first draft, the reason it failed, and the number of attempts are all gone.

---
## 2. LangGraph implementation

LangGraph puts the loop *inside* the framework. More setup, but the retry logic becomes part
of the graph and is therefore visualisable, streamable, and resumable.

In [3]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END


class DraftState(TypedDict):
    topic: str
    draft: str
    verdict: str
    attempts: int


def write_node(state: DraftState) -> dict:
    text = write.invoke({"topic": state["topic"]})
    return {"draft": text, "attempts": state.get("attempts", 0) + 1}


def check_node(state: DraftState) -> dict:
    return {"verdict": check.invoke(
        {"draft": state["draft"], "standard": STANDARD}
    ).strip().upper()}


def revise_node(state: DraftState) -> dict:
    text = revise.invoke({"draft": state["draft"], "standard": STANDARD})
    return {"draft": text, "attempts": state["attempts"] + 1}


def route(state: DraftState) -> Literal["revise", "__end__"]:
    if state["verdict"].startswith("PASS") or state["attempts"] >= 3:
        return "__end__"
    return "revise"


builder = StateGraph(DraftState)
builder.add_node("write", write_node)
builder.add_node("check", check_node)
builder.add_node("revise", revise_node)

builder.add_edge(START, "write")
builder.add_edge("write", "check")
builder.add_conditional_edges("check", route, {"revise": "revise", "__end__": END})
builder.add_edge("revise", "check")           # the loop

draft_graph = builder.compile()
print(draft_graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	write(write)
	check(check)
	revise(revise)
	__end__([<p>__end__</p>]):::last
	__start__ --> write;
	check -.-> __end__;
	check -.-> revise;
	revise --> check;
	write --> check;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [4]:
start = time.time()

for step in draft_graph.stream({"topic": TOPIC, "attempts": 0}, stream_mode="updates"):
    for node, update in step.items():
        print(f"[{node}] {str(update)[:90]}")

final_state = draft_graph.invoke({"topic": TOPIC, "attempts": 0})
timings["LangGraph"] = time.time() - start

print("\n--- LANGGRAPH RESULT ---\n", final_state["draft"])
print("\nAttempts:", final_state["attempts"], "| Verdict:", final_state["verdict"])

[write] {'draft': "If your claim was denied because you didn't get prior authorisation, here’s wha


[check] {'verdict': 'PASS'}



--- LANGGRAPH RESULT ---
 If your claim was denied because you didn't get prior authorisation, here’s what to do: First, read the denial letter carefully. Note the reason for the denial and any instructions provided. You must request a review of your claim within 60 days of receiving the denial letter. To do this, call our customer service at 1-800-555-HELP and ask for a prior authorisation review. Be ready to provide your member ID and the claim number. A representative will guide you through the next steps. If you need help finding the denial letter or understanding the process, don’t hesitate to ask. We’re here to help you get the coverage you need.

Attempts: 1 | Verdict: PASS


**What that felt like:** roughly three times the code of the LangChain version. In
exchange you got a diagram of the workflow, a live stream of every node, an explicit attempt
counter in state, and — with a checkpointer added — the ability to resume or replay any step.

That attempt counter is not a debugging nicety in this domain. It is the answer to "how many
times did the automated reviewer reject this before it went out, and what changed each time?"

---
## 3. CrewAI implementation

CrewAI reframes the same problem as two colleagues: a writer who drafts and a reviewer who
checks against the standard. There is no explicit loop at all — the reviewer's task simply
produces the corrected version.

In [5]:
from crewai import Agent, Task, Crew, Process, LLM

crew_llm = LLM(model=f"bedrock/{CHAT_MODEL_ID}", temperature=0.2, max_tokens=1024)

crew_writer = Agent(
    role="Member Communications Writer",
    goal="Write plain-language explanations members can act on",
    backstory=(
        "You write for stressed people reading on a phone. You state deadlines as numbers of "
        "days, never as 'promptly'. You never give clinical advice."
    ),
    llm=crew_llm,
    verbose=False,
    allow_delegation=False,
)

crew_reviewer = Agent(
    role="Member Communications Reviewer",
    goal="Ensure every member communication meets the plan's communication standard",
    backstory=(
        "You reject vague instructions, missing deadlines, unexplained jargon and anything "
        "that reads as medical advice. You rewrite rather than merely comment."
    ),
    llm=crew_llm,
    verbose=False,
    allow_delegation=False,
)

write_task = Task(
    description=(
        "Write a 120-word plain-language explanation for a member about {topic}. "
        "Say what they must do, state any deadline explicitly, and say how to reach a person."
    ),
    expected_output="A 120-word explanation in three short paragraphs.",
    agent=crew_writer,
)

review_task = Task(
    description=(
        "Review the draft against this standard and return a corrected version:\n" + STANDARD
    ),
    expected_output="The corrected 120-word explanation, ready to send.",
    agent=crew_reviewer,
    context=[write_task],
)

start = time.time()
crew_result = Crew(
    agents=[crew_writer, crew_reviewer],
    tasks=[write_task, review_task],
    process=Process.sequential,
    verbose=False,
).kickoff(inputs={"topic": TOPIC})
timings["CrewAI"] = time.time() - start

print("--- CREWAI RESULT ---\n", crew_result.raw)
print("\nTokens:", crew_result.token_usage)

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: f26cd89e-f253-4cdd-aa0d-0bb3cf3c096d                                  │
│                                                                                                                 │
│ 🔗 View here:                                                                                                   │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/f26cd89e-f253-4cdd-aa0d-0bb3cf3c096d?access_code=TRA │
│ CE-2a4181b198                                                                                                   │
│ 🔑 Access Code: TRACE-2a4181b198                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

--- CREWAI RESULT ---
 If your claim was denied due to lack of prior authorization, please follow these steps to resolve the issue. First, carefully review your denial letter for specific reasons and instructions. Within 7 days, contact your healthcare provider to obtain the necessary prior authorization. They will guide you on submitting the required information to us. After securing the authorization, reach out to our Member Services team within 14 days. You can contact us by phone at 1-800-123-4567 or via email at help@insurancecompany.com. When you contact us, please provide your claim number and confirm that you have obtained the prior authorization. Our team will review your case and inform you of the next steps. It is crucial to act promptly to address the denial. If you do not appeal within 14 days, your claim may be considered final, and you may lose the opportunity to have it reconsidered.

Tokens: total_tokens=1030 prompt_tokens=654 cached_prompt_tokens=0 completion_tokens=3

**What that felt like:** the least code and by far the most readable intent — you can
hand these agent definitions to a non-engineer and they will understand them. But you gave up
control: there is no verdict variable you can branch on, no attempt counter, and no guarantee
the reviewer actually applied the standard rather than lightly paraphrasing.

Notice what that costs you here specifically. You cannot prove the communication standard was
applied. The reviewer agent *says* it reviewed; there is no PASS or FAIL you can log, no record
of what failed, and no way to block release on a failed check. For an internal draft that a
human approves, that is fine. For anything automated, it is not.

---
## 4. Side-by-side

In [6]:
print(f"{'Framework':<12} {'Seconds':>9}")
print("-" * 22)
for name, seconds in timings.items():
    print(f"{name:<12} {seconds:>9.1f}")

Framework      Seconds
----------------------
LangChain          1.6
LangGraph          3.4
CrewAI             3.7


> Timings vary run to run — treat them as a rough ordering, not a benchmark. What is
> consistent is the *shape*: LangChain makes the fewest model calls, CrewAI usually the most,
> because every agent handoff carries reasoning overhead.

### 4.1. Capability comparison

| | **LangChain** | **LangGraph** | **CrewAI** |
|---|---|---|---|
| **Core abstraction** | Runnable / chain | Graph of nodes + state | Team of role-playing agents |
| **Control flow** | Linear pipe, or an agent loop | Any graph: loops, branches, parallel | Sequential or manager-led delegation |
| **Who decides the path** | You (chain) or the model (agent) | You, explicitly, in code | The manager agent or task order |
| **State** | Passed along the pipe | First-class, typed, checkpointed | Implicit in task context |
| **Cycles / retries** | Written in Python around the chain | Native — a conditional edge | Not directly expressible |
| **Persistence & resume** | Not built in | Checkpointers, time travel | Limited |
| **Observability** | Callbacks, LangSmith | Graph diagram, step streaming, state history | Verbose logs, token usage |
| **Human-in-the-loop** | Manual | Native (interrupts) | Limited |
| **Auditability of a decision** | Final output only | Full state history, replayable | Verbose log, not structured |
| **Lines of code (this task)** | Fewest | Most | Middle |
| **Learning curve** | Low | Medium-high | Low |
| **Best at** | Composing model calls | Reliable, auditable workflows | Role-based collaborative work |
| **Weakest at** | Anything cyclic or stateful | Simple tasks — it is overkill | Deterministic control and auditing |

### 4.2. How they relate

They are layers, not rivals:

```
   CrewAI            <- role-based teams, highest level of abstraction
      |
   LangGraph         <- explicit stateful graphs, full control of flow
      |
   LangChain         <- models, prompts, tools, retrievers, chains
      |
   Amazon Bedrock    <- the foundation models
```

`create_agent()` from notebook 1 **is** a LangGraph graph underneath. CrewAI agents call the
same Bedrock models through the same kind of interface. Learning them in this order is
learning one stack, not three products.

### 4.3. Which one should you use?

Work down this list and stop at the first match:

1. **One model call, or a fixed sequence of them?** → LangChain chain. Do not over-engineer.
2. **The model needs to pick tools, but the flow is a simple loop?** → LangChain `create_agent`.
3. **You need retries, branching, parallelism, resume, or an audit trail?** → LangGraph.
4. **A human would solve this by handing work between specialists?** → CrewAI.
5. **A regulated or high-stakes workflow where every step must be reproducible?** → LangGraph,
   with a checkpointer, every time.

And the combination that shows up most often in production: **LangChain for the components,
LangGraph for the orchestration.** CrewAI is chosen when the team metaphor genuinely matches
the problem — content pipelines, research workflows, multi-perspective review — and when
readable agent definitions matter more than deterministic control.

### 4.4. What each one costs you

| | Cost driver | Typical failure mode |
|---|---|---|
| LangChain | One call per step | Silently wrong output — no built-in verification step |
| LangGraph | One call per node, plus retries | Over-engineering; graphs that are harder to read than the logic they encode |
| CrewAI | Manager reasoning + every handoff | Agents that agree with each other and produce confident, generic output |

The practical implication: on Bedrock you are billed per token, so **the framework you choose
changes your bill**. A hierarchical CrewAI run on this task can use several times the tokens of
the equivalent LangChain chain. That is a fair trade when the collaboration adds real value,
and pure waste when it does not.

### 4.5. Choosing for a healthcare workload specifically

Rule 5 above does most of the work in this domain, so it is worth being concrete about which
parts of a member-facing system land where:

| Component | Framework | Why |
|---|---|---|
| Plan document retrieval, cost calculations, notice rewriting | **LangChain** | Fixed sequences of model and tool calls; nothing to branch on |
| Inbound member message routing, including the clinical scope check | **LangGraph** | The guardrail must be structural and the path must be replayable |
| Prior authorisation and appeal workflows | **LangGraph** | Retries, human interrupts, and a state history you can produce on request |
| Drafting member communications for human approval | **CrewAI** or LangChain | A human is the last step, so path variability is acceptable |
| Pre-launch design review of a new feature | **CrewAI** | Multi-perspective review is genuinely what a team of specialists does |

The dividing line is not complexity. It is **whether a human sees the output before a member
does**. Where a human approves, optimise for drafting speed and readability. Where the output
reaches a member directly, optimise for a workflow you can draw, replay and evidence.

---
## Summary

- **LangChain** composes model calls. Reach for it first.
- **LangGraph** owns control flow — loops, branches, state, resume, audit.
- **CrewAI** models collaboration between specialists.
- They stack: CrewAI and LangGraph both sit on LangChain-style components, which sit on Bedrock.
- Choose by the **shape of the control flow**, not by which framework is newest.
- In healthcare, add one more question: does a human see this before the member does?

You have now built, on Amazon Bedrock: a chain, a tool-calling agent, a memory-backed member
conversation, a full RAG pipeline with page citations over a real benefits handbook, a custom
state graph with a structural scope guardrail and time travel, and a multi-agent crew that
reviews a design the way a governance committee would.

### Where to go next

- Point the RAG pipeline at your own evidence of coverage or provider manual instead of the
  handbook, and see how much of notebook 3 changes. It should be one cell.
- Replace Chroma with Amazon OpenSearch Serverless or pgvector on Amazon Aurora.
- Add Amazon Bedrock Guardrails on top of the scope router from notebook 4 and compare how the
  two layers behave on the same mixed clinical/benefits message.
- Add a human-in-the-loop interrupt to the notebook 4 inquiry graph so a representative approves
  every reply before it is sent.
- Build the evaluation set this lab keeps referring to: twenty real member question types, with
  the correct routing and the required citations, run on every change.